# <font color='orange'>**RapidRelief AI — Sprint 2 & 3: Transferencia de Aprendizaje**</font>

**Metodología:** MobileNetV2 + `include_top=False` + cabeza personalizada + fine-tuning  
**Dataset:** Clothing Small (fotos reales de prendas)  
**Meta:** `val_accuracy ≥ 0.90`

| Versión | Loss | Capas fine-tuning | shirt F1 | Val acc |
|---------|------|-------------------|----------|---------|
| v2 | CrossEntropy | 54 | — | 0.87 |
| v3 | CrossEntropy + cw | 30 | 0.53 | 0.8886 |
| **v4** | **Focal Loss + cw** | **60** | **→ ?** | **→ ?** |

**Cambios v4:** Focal Loss (γ=2) · 60 capas · shirt×2.5 · outwear×1.5 · TTA · patience aumentada

## 1. Importar el modelo de interés

Igual que en el notebook de clase — MobileNetV2 con `TF_USE_LEGACY_KERAS`.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
%matplotlib inline

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Rutas del proyecto
BASE_DIR    = '/content/drive/MyDrive/RapidReliefAI'
CLOTH_TRAIN = BASE_DIR + '/Datasets/clothing_small/train'
CLOTH_VAL   = BASE_DIR + '/Datasets/clothing_small/validation'
CLOTH_TEST  = BASE_DIR + '/Datasets/clothing_small/test'
MODELS_DIR  = BASE_DIR + '/Models'

# Parametros — mismos que en clase
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
N_CLASES   = 10

# Etiquetas finales (orden alfabetico = orden que asigna flow_from_directory)
CLASES = ['dress', 'hat', 'longsleeve', 'outwear', 'pants',
          'shirt', 'shoes', 'shorts', 'skirt', 't-shirt']
print('Configuracion lista')
print(f'Clases ({N_CLASES}):', CLASES)

## 2. Importar la ruta del DATASET

Mismo patrón que en clase: `ImageDataGenerator` + `flow_from_directory` + `preprocessing_function=preprocess_input`.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=15.0,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=[0.80, 1.20],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

print('Cargando datos...')
train_data = train_datagen.flow_from_directory(
    CLOTH_TRAIN,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

val_data = val_datagen.flow_from_directory(
    CLOTH_VAL,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=42
)

test_data = val_datagen.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

In [ ]:
# Verificar forma del batch — igual que en clase
imgs, labels = next(train_data)
print(f'Forma imagenes: {imgs.shape}')   # (32, 224, 224, 3)
print(f'Forma etiquetas: {labels.shape}') # (32, 10)
print('Indice de clases:', train_data.class_indices)

In [ ]:
# Visualizar batch — misma funcion que en clase
def plotImages(images_arr):
    fig, axes = plt.subplots(1, 10, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(images_arr, axes):
        # Revertir preprocess_input para visualizacion
        img_vis = (img + 1.0) / 2.0
        ax.imshow(np.clip(img_vis, 0, 1))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

plotImages(imgs)
print(labels[:10])
# Etiquetas: 0=dress 1=hat 2=longsleeve 3=outwear 4=pants 5=shirt 6=shoes 7=shorts 8=skirt 9=t-shirt

## 3. Cargar el modelo pre-entrenado

Igual que en clase: `include_top=False` para no cargar las capas densas de ImageNet,  
y `trainable = False` para congelar todos los pesos del modelo base.

In [ ]:
# Al cargar un modelo, el argumento include_top se establece en False
# para que las capas de salida densas del modelo no sean cargadas,
# permitiendo agregar y entrenar una nueva capa de salida
pre_trained_model = MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

In [ ]:
# Se debe congelar el modelo base — no se ajustaran los pesos del modelo pre-entrenado
pre_trained_model.trainable = False

### <font color='darkviolet'>**Cabeza personalizada — API Funcional de Keras**</font>

Correcciones respecto a v1:
- `GlobalAveragePooling2D` en lugar de `Flatten` — reduce parámetros de 128M a ~655K
- `Dropout(0.3)` para regularización y reducir sobreajuste
- Eliminada `Dense(16, sigmoid)` — `sigmoid` es multi-etiqueta, no multi-clase

In [ ]:
pre_trained_model.output

In [ ]:
pre_trained_model.input

In [ ]:
pre_trained_model.summary()

In [ ]:
# GAP → Dense(512) → Dropout → Dense(10)
# GlobalAveragePooling2D: promedia el tensor 7×7×1280 a 1280 features (en lugar de Flatten=62720)
x = GlobalAveragePooling2D()(pre_trained_model.output)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)   # era 0.3 — cierra la brecha train-val de 0.0515
output = Dense(N_CLASES, activation='softmax')(x)

In [ ]:
modelo = Model(inputs=pre_trained_model.input, outputs=output)
modelo.summary()

In [ ]:
def focal_loss(gamma=2.0):
    """
    Focal Loss (Lin et al., ICCV 2017).
    Combinar con class_weight (actúa como alpha) para el desbalance por clase.
    gamma=2.0: óptimo para imbalance moderado (ratio shirt:shoes ≈ 1:2.8).
    """
    def loss_fn(y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce      = -y_true * tf.math.log(y_pred)
        p_t     = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        weight  = tf.pow(1.0 - p_t, gamma)
        return tf.reduce_mean(weight * tf.reduce_sum(ce, axis=-1, keepdims=True))
    return loss_fn

GAMMA = 2.0
print(f'Focal Loss definida — gamma={GAMMA}')
print('Efecto: clases difíciles (shirt, longsleeve) reciben gradiente amplificado')
print('        clases fáciles (shoes, pants) reducen su dominio del gradiente')

### <font color='darkviolet'>**Función de pérdida — Focal Loss (v4)**</font>

**Cambio respecto a v3:** `categorical_crossentropy` → **Focal Loss** (Lin et al., ICCV 2017).

La cross-entropy trata todos los ejemplos por igual; la focal loss aplica un factor `(1−p_t)^γ` que **reduce el peso de los ejemplos ya bien clasificados** (p_t alto) y concentra el gradiente en los difíciles. Con γ=2 y los class weights como alpha:

- **`shirt` (p_t bajo, confusión alta)** → recibe gradiente amplificado
- **`shoes` (p_t alto, ya converge bien)** → su contribución se reduce para no dominar el gradiente

In [ ]:
modelo.compile(
    loss=focal_loss(gamma=GAMMA),
    optimizer=Adam(learning_rate=1e-3),
    metrics=['accuracy']
)
print(f'Modelo compilado — Phase A | base congelada | Adam lr=1e-3 | Focal Loss γ={GAMMA}')

## 4. Fase A — Entrenar la cabeza (base congelada)

Se entrena solo la cabeza nueva con la base MobileNetV2 completamente congelada.  
`class_weight` compensa el desbalance entre clases (hat/skirt: 12 imgs vs shoes: 73).

In [ ]:
# Class weights — compensar desbalance entre clases
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(N_CLASES),
    y=train_data.classes
)
cw = dict(enumerate(class_weights_arr))

# Ajuste basado en análisis de matriz de confusión v3:
# shirt (idx=5): F1=0.53, precision=0.48 — confusión severa con t-shirt y longsleeve
# longsleeve (idx=2): recall=0.74 — falsos negativos predichos como shirt
# outwear (idx=3): recall=0.74 — subestimado, confusión con longsleeve
cw[2] *= 1.5   # longsleeve
cw[3] *= 1.5   # outwear
cw[5] *= 2.5   # shirt

print('Class weights ajustados (v4):')
for k, v in cw.items():
    print(f'  {CLASES[k]:<12} {v:.3f}')

os.makedirs(MODELS_DIR, exist_ok=True)

callbacks_A = [
    ModelCheckpoint(
        MODELS_DIR + '/rapidrelief_phaseA_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,          # era 6 — más margen para superar mesetas temporales
        min_delta=0.001,      # solo contar mejoras reales (>0.1%), ignorar ruido
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,           # era 3
        min_lr=1e-6,
        verbose=1
    ),
]

history_A = modelo.fit(
    train_data,
    validation_data=val_data,
    epochs=40,                # era 25 — más margen para que focal loss converja
    callbacks=callbacks_A,
    class_weight=cw
)

## 5. Guardar modelo Phase A

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)
modelo.save(MODELS_DIR + '/rapidrelief_mobilenetv2_phaseA.keras')
print('Modelo Phase A guardado:', MODELS_DIR + '/rapidrelief_mobilenetv2_phaseA.keras')

## 6. Curvas de entrenamiento — Phase A

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_A.history['accuracy'], label='Train')
plt.plot(history_A.history['val_accuracy'], 'r--', label='Validación')
plt.title('Phase A — Exactitud')
plt.ylabel('Exactitud')
plt.xlabel('Épocas')
plt.axhline(0.90, color='green', linestyle=':', linewidth=1, label='KPI 90%')
plt.legend()
plt.tight_layout()
plt.show()

gap = max(history_A.history['accuracy']) - max(history_A.history['val_accuracy'])
print(f'Val accuracy máx: {max(history_A.history["val_accuracy"]):.4f}')
print(f'Brecha train-val: {gap:.4f} (objetivo < 0.05)')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_A.history['loss'], label='Train')
plt.plot(history_A.history['val_loss'], 'r--', label='Validación')
plt.title('Phase A — Pérdida')
plt.ylabel('Loss')
plt.xlabel('Épocas')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Fase B — Fine-tuning (últimas 60 capas de MobileNetV2)

Con la cabeza ya entrenada, descongelamos las **últimas 60 capas** del backbone (v3: 30) y ajustamos con LR más bajo (`1e-5`) para refinar representaciones sin destruir los pesos de ImageNet.

**Cambios v4 respecto a v3:**
- **60 capas** en lugar de 30 — investigación (Sandler et al. 2018): capas 95–155 contienen texturas de tela y partes de prendas necesarias para distinguir shirt vs t-shirt
- **LR = 1e-5** (bajado de 2e-5) — mayor nº de parámetros activos requiere paso más conservador
- **patience = 20, min_delta = 0.001** — evitar que el early stopping corte antes de que la focal loss termine de redistribuir el gradiente entre clases difíciles
- **epochs ≤ 60** — margen suficiente para convergencia lenta del fine-tuning profundo

In [ ]:
# Descongelar las últimas 60 capas de MobileNetV2 (v3 usaba 30 — insuficiente)
# Howard & Ruder (2018): capas más profundas contienen features de alto nivel (partes de objetos)
# MobileNetV2 tiene ~155 capas; capas 95–155 contienen texturas y ensamblaje de prendas
# Para clasificación fina de ropa (collar vs cuello redondo) se necesitan ≥60 capas
# LR reducido a 1e-5 (vs 2e-5) para compensar mayor número de parámetros ajustables
pre_trained_model.trainable = True
for layer in pre_trained_model.layers[:-60]:
    layer.trainable = False

trainable_count = sum(1 for l in pre_trained_model.layers if l.trainable)
print(f'Capas entrenables en base: {trainable_count} de {len(pre_trained_model.layers)}')
print(f'(v3: 30 capas → v4: 60 capas — mayor profundidad para features finos de ropa)')

modelo.compile(
    loss=focal_loss(gamma=GAMMA),
    optimizer=Adam(learning_rate=1e-5),
    metrics=['accuracy']
)
print(f'Modelo recompilado — Phase B | 60 capas | Adam lr=1e-5 | Focal Loss γ={GAMMA}')

callbacks_B = [
    ModelCheckpoint(
        MODELS_DIR + '/rapidrelief_phaseB_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=20,          # era 10 — fine-tuning profundo mejora lentamente
        min_delta=0.001,      # solo mejoras reales (>0.1%), ignorar oscilaciones
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,           # era 4
        min_lr=1e-7,
        verbose=1
    ),
]

history_B = modelo.fit(
    train_data,
    validation_data=val_data,
    epochs=60,                # era 40 — dar margen al fine-tuning profundo
    callbacks=callbacks_B,
    class_weight=cw
)

In [ ]:
modelo.save(MODELS_DIR + '/rapidrelief_mobilenetv2_v4.keras')
print('Modelo final (Phase B) guardado:', MODELS_DIR + '/rapidrelief_mobilenetv2_v4.keras')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_B.history['accuracy'], label='Train')
plt.plot(history_B.history['val_accuracy'], 'r--', label='Validación')
plt.title('Phase B — Exactitud (fine-tuning)')
plt.ylabel('Exactitud')
plt.xlabel('Épocas')
plt.axhline(0.90, color='green', linestyle=':', linewidth=1, label='KPI 90%')
plt.legend()
plt.tight_layout()
plt.show()

gap_B = max(history_B.history['accuracy']) - max(history_B.history['val_accuracy'])
print(f'Val accuracy Phase B máx: {max(history_B.history["val_accuracy"]):.4f}')
print(f'Brecha train-val Phase B: {gap_B:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_B.history['loss'], label='Train')
plt.plot(history_B.history['val_loss'], 'r--', label='Validación')
plt.title('Phase B — Pérdida (fine-tuning)')
plt.ylabel('Loss')
plt.xlabel('Épocas')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Evaluación — Matriz de Confusión y Reporte por Clase

Herramienta clave para identificar solapamientos entre clases similares (ej. shirt vs t-shirt).

In [ ]:
# TTA — Test-Time Augmentation
# Literatura: +1.5–2% accuracy en clases con bajo recall (longsleeve, outwear)
# Principio: prendas simétricas (camisas, pantalones) son invariantes al flip horizontal
# Soft voting: promediar logits es más estable que votar la clase más frecuente

test_datagen_flip = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True
)
test_data_flip = test_datagen_flip.flow_from_directory(
    CLOTH_TEST,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    seed=42
)

# Predicciones originales
test_data.reset()
preds_orig = modelo.predict(test_data, verbose=1)

# Predicciones con flip horizontal
test_data_flip.reset()
preds_flip = modelo.predict(test_data_flip, verbose=1)

# Soft voting: promedio de logits
predicciones_tta = (preds_orig + preds_flip) / 2.0

y_pred     = np.argmax(preds_orig, axis=1)
y_pred_tta = np.argmax(predicciones_tta, axis=1)
y_true     = test_data.classes

acc_base = np.mean(y_pred == y_true)
acc_tta  = np.mean(y_pred_tta == y_true)
print(f'Accuracy sin TTA:  {acc_base:.4f}')
print(f'Accuracy con TTA:  {acc_tta:.4f}  (Δ = {acc_tta - acc_base:+.4f})')
print(f'Total predicciones: {len(y_pred_tta)}')

# Usar TTA para el reporte final
predicciones = predicciones_tta

In [ ]:
# Matriz de Confusion con Seaborn (igual que en brief.md)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASES,
    yticklabels=CLASES
)
plt.title('Matriz de Confusion — RapidRelief AI (MobileNetV2)', fontsize=13)
plt.ylabel('Etiqueta real')
plt.xlabel('Etiqueta predicha')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Reporte completo por clase (precision, recall, F1)
print('Reporte de clasificacion por clase:')
print(classification_report(y_true, y_pred, target_names=CLASES))

## 9. Inferencia con imágenes individuales

Verifica predicciones sobre imágenes reales del conjunto de test.

In [ ]:
from tensorflow.keras.preprocessing import image

def predecir_prenda(img_path):
    img = image.load_img(img_path, target_size=IMAGE_SIZE)
    X = image.img_to_array(img)
    X = np.expand_dims(X, axis=0)
    X = preprocess_input(X)
    preds = modelo.predict(X, verbose=0)
    clase_idx = np.argmax(preds, axis=1)[0]
    clase_nombre = CLASES[clase_idx]
    confianza = preds[0][clase_idx]

    print(f'Prediccion: {clase_nombre} (confianza: {confianza:.2%})')
    print(f'Vector de probabilidades: {preds[0].round(3)}')

    plt.imshow(img)
    plt.title(f'Prediccion: {clase_nombre} ({confianza:.1%})')
    plt.axis('off')
    plt.show()
    return clase_nombre

In [ ]:
# Probar con una imagen del conjunto de test
# Modificar img_path con cualquier imagen real de tus carpetas de test
img_path = CLOTH_TEST + '/dress/' + os.listdir(CLOTH_TEST + '/dress/')[0]
predecir_prenda(img_path)

## 10. Exportar a TF Lite (para app móvil)

Cuantización INT8 — reduce el modelo de ~14 MB a ~4 MB sin pérdida significativa de accuracy.

In [ ]:
import tensorflow as tf

# Conjunto representativo para calibración INT8 (100 batches del test set)
def representative_dataset():
    test_data.reset()
    count = 0
    for imgs, _ in test_data:
        for img in imgs:
            yield [np.expand_dims(img.astype(np.float32), axis=0)]
            count += 1
            if count >= 100:
                return

converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

tflite_model = converter.convert()

tflite_path = MODELS_DIR + '/rapidrelief_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'Modelo TFLite INT8 guardado: {tflite_path}')
print(f'Tamaño: {size_mb:.1f} MB  (objetivo: ≤ 6 MB)')

In [ ]:
# Guardar archivo de etiquetas para la app
labels_path = MODELS_DIR + '/labels.txt'
with open(labels_path, 'w') as f:
    for clase in CLASES:
        f.write(clase + '\n')

print('Archivo de etiquetas guardado:')
for i, clase in enumerate(CLASES):
    print(f'  {i}: {clase}')

## 11. Resumen final

In [ ]:
val_acc_A   = max(history_A.history['val_accuracy'])
val_acc_B   = max(history_B.history['val_accuracy'])
train_acc_B = max(history_B.history['accuracy'])
gap         = train_acc_B - val_acc_B

print('=== RESUMEN FINAL — v4 ===')
print(f'Arquitectura:      MobileNetV2 → GAP → Dense(512,relu) → Dropout(0.4) → Dense(10,softmax)')
print(f'Optimizador A:     Adam lr=1e-3  |  Optimizador B: Adam lr=1e-5')
print(f'Loss:              Focal Loss (γ={GAMMA}) + class weights (actúa como alpha)')
print(f'Augmentation:      rotation=20, shift=0.15, shear=15, zoom=0.20, brightness=[0.80,1.20], fill=reflect')
print(f'Class weights:     balanced + longsleeve×1.5 + outwear×1.5 + shirt×2.5')
print(f'Fine-tuning:       últimas 60 capas (v3: 30) | patience_B=20 | min_delta=0.001 | epochs_B≤60')
print(f'TTA:               flip horizontal (soft voting, 2 pasadas)')
print(f'Val accuracy Phase A: {val_acc_A:.4f}')
print(f'Val accuracy Phase B: {val_acc_B:.4f}')
print(f'Brecha train-val:  {gap:.4f}  (objetivo < 0.05)')
print(f'KPI (>=0.90):      {"✓ CUMPLIDO" if val_acc_B >= 0.90 else "✗ NO cumplido"}')
print(f'Modelo final:      {MODELS_DIR}/rapidrelief_mobilenetv2_v4.keras')
print(f'TFLite INT8:       {MODELS_DIR}/rapidrelief_model.tflite ({size_mb:.1f} MB)')